# Brainclip Colab Voice Runtime — Fish S2-Pro

This notebook sets up an isolated virtual environment, installs Fish S2-Pro and pinned dependencies, starts a FastAPI voice runtime, and exposes it via Cloudflare Tunnel.

**Model:** `fishaudio/s2-pro` — 5B parameter dual-autoregressive TTS with inline emotion/prosody control via `[tag]` syntax. Zero-shot voice cloning from reference audio.

**GPU requirement:** T4 (16GB VRAM) or better. S2-Pro BF16 weights need ~10GB VRAM. Remaining headroom for KV cache and audio buffers.

## Why a virtual environment

Colab base packages change over time. We pin versions in a local venv so dependency drift does not silently break the voice server.

In [1]:
from pathlib import Path
import os
import sys

APP_ROOT = Path('/content')
VENV_DIR = APP_ROOT / 'venv'
print('Python version:', sys.version)
if sys.version_info < (3, 10):
    raise RuntimeError('Brainclip runtime expects Python 3.10 or newer.')


Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [7]:
%%bash
set -e
cd /content

# Clone / update repo
if [ ! -d "notebooks" ]; then
  git clone https://github.com/harshal0704/Brainclip.git temp_repo
  mv temp_repo/* . || true
  rm -rf temp_repo
fi

# Create venv without pip to avoid ensurepip errors
rm -rf /content/venv
python3 -m venv /content/venv --without-pip
source /content/venv/bin/activate

# Manually install pip
curl -sS https://bootstrap.pypa.io/get-pip.py | python

# Install core build tools
python -m pip install --upgrade 'pip<25' 'setuptools<75' 'wheel<0.46'

# 1. Install/Upgrade core ML stack to match vLLM 0.8.4 requirements
python -m pip install --upgrade 'torch==2.6.0' 'torchaudio==2.6.0' 'huggingface-hub[hf_xet]>=0.30.0'

# 2. Install vllm
python -m pip install vllm==0.8.4

# 3. Remove conflicting pins from requirements.txt
sed -i '/transformers==/d' /content/notebooks/requirements.txt
sed -i '/vllm==/d' /content/notebooks/requirements.txt
sed -i '/torch==/d' /content/notebooks/requirements.txt
sed -i '/torchaudio==/d' /content/notebooks/requirements.txt
sed -i '/huggingface-hub==/d' /content/notebooks/requirements.txt

# 4. Install remaining requirements
python -m pip install -r /content/notebooks/requirements.txt

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Using cached pip-24.3.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached setuptools-74.1.3-py3-none-any.whl.metadata (6.7 kB)
  Using cached wheel-0.45.1-py3-none-any.whl.metadata (2.3 kB)
Using cached pip-24.3.1-py3-none-any.whl (1.8 MB)
Using cached setuptools-74.1.3-py3-none-any.whl (1.3 MB)
Using cached wheel-0.45.1-py3-none-any.whl (72 kB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1

  Using cached torch-2.6.0-cp312-cp312-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchaudio-2.6.0-cp312-cp312-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached huggingface_hub-1.7.2-py3-none-any.whl.metadata (13 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networ

## Download Fish S2-Pro model (~9 GB)

This cell downloads the sharded S2-Pro checkpoint from HuggingFace. It only runs once per session — subsequent runs skip re-download.

**Estimated time:** 5–15 minutes on fast Colab internet.

In [6]:
%%bash
set -e
source /content/venv/bin/activate

MODEL_DIR="/content/models/s2-pro"

# Skip if already fully downloaded
if [ -d "$MODEL_DIR" ] && [ "$(ls -1 $MODEL_DIR | wc -l)" -gt 5 ]; then
    echo "Model already present — skipping download."
    ls -lh "$MODEL_DIR"
else
    mkdir -p "$MODEL_DIR"
    huggingface-cli download fishaudio/s2-pro \
        --local-dir "$MODEL_DIR" \
        --local-dir-use-symlinks False
    echo "Download complete."
    ls -lh "$MODEL_DIR"
fi


/content/models/s2-pro
Download complete.
total 11G
-rw-r--r-- 1 root root 4.1K Mar 22 08:58 chat_template.jinja
-rw-r--r-- 1 root root 1.8G Mar 22 08:58 codec.pth
-rw-r--r-- 1 root root 1.9K Mar 22 08:58 config.json
-rw-r--r-- 1 root root  11K Mar 22 08:58 LICENSE.md
-rw-r--r-- 1 root root 4.7G Mar 22 08:59 model-00001-of-00002.safetensors
-rw-r--r-- 1 root root 3.9G Mar 22 08:59 model-00002-of-00002.safetensors
-rw-r--r-- 1 root root  32K Mar 22 08:58 model.safetensors.index.json
-rw-r--r-- 1 root root 3.4M Mar 22 08:58 overview.png
-rw-r--r-- 1 root root 5.0K Mar 22 08:58 README.md
-rw-r--r-- 1 root root 100K Mar 22 08:58 special_tokens_map.json
-rw-r--r-- 1 root root 841K Mar 22 08:58 tokenizer_config.json
-rw-r--r-- 1 root root  12M Mar 22 08:58 tokenizer.json


/content/venv/lib/python3.12/site-packages/huggingface_hub/commands/download.py:139: FutureWarning: Ignoring --local-dir-use-symlinks. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]Downloading 'config.json' to '/content/models/s2-pro/.cache/huggingface/download/8_PA_wEVGiVa2goH2H4KQOQpvVY=.5214fbf56c44b20d00b1d5654d68b3b1f9ec9ea5.incomplete'
Download complete. Moving file to /content/models/s2-pro/config.json
Download complete. Moving file to /content/models/s2-pro/README.md
Download complete. Moving file to /content/models/s2-pro/chat_template.jinja
Download complete. Moving file to /content/models/s2-pro/LICENSE.md
Download complete. Moving file to /content/models/s2-pro/.gitattributes
Fetching 13 files:   8%|▊         | 1/13 [00:00<00:03,  3.05it/s]Downloading 'overview.png' to '/content/models/s2-pro/.cache/huggingface/download/7mPi-8xkJLdBhz3c9ZSMvUdMRVw=.f77da18e7d3cf59182fb714fc6c1bc526e7

## Verify GPU + VRAM

Check that CUDA is available and we have enough VRAM for S2-Pro (~10GB BF16).

In [8]:
%%bash
set -e
source /content/venv/bin/activate
python -c "
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')
if torch.cuda.is_available():
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM total: {mem_total:.1f} GB')
    import torch.cuda
    mem_free, mem_allocated = torch.cuda.mem_get_info()
    print(f'VRAM free before load: {mem_free/1e9:.1f} GB')
    print(f'Recommended: 12+ GB free for S2-Pro.')
else:
    print('WARNING: No GPU detected. Fish S2-Pro requires CUDA.')
"


CUDA available: True
GPU: Tesla T4
VRAM total: 15.6 GB
VRAM free before load: 15.5 GB
Recommended: 12+ GB free for S2-Pro.


## Start FastAPI server + Cloudflare Tunnel

Run this cell to start the FastAPI voice runtime in the background, then start a Cloudflare Tunnel to expose it publicly. Copy the printed URL into Brainclip settings.

**Keep this cell running** — stopping it will terminate the server and tunnel.

In [13]:
%%bash
set -e
source /content/venv/bin/activate
export BRAINCLIP_APP_ROOT=/content
export PYTHONPATH=$PYTHONPATH:/content

# Kill any previous server on port 8000
pkill -f "uvicorn.*brainclip_colab_server" 2>/dev/null || true
sleep 1

# Verify vllm installation in venv
/content/venv/bin/python -c "import vllm; print('vLLM version:', vllm.__version__)" || (echo "vLLM not found in venv!" && exit 1)

# Start FastAPI server in background using the VENV python directly
# This ensures the process always uses the venv packages
nohup /content/venv/bin/python -m uvicorn notebooks.brainclip_colab_server:app \
    --app-dir /content \
    --host 0.0.0.0 \
    --port 8000 \
    > /content/uvicorn.log 2>&1 &

sleep 10

# Verify server is up
curl -s http://127.0.0.1:8000/health | python -m json.tool || cat /content/uvicorn.log

INFO 03-22 09:18:53 [__init__.py:239] Automatically detected platform cuda.
vLLM version: 0.8.4
{
    "status": "ok",
    "model": "fishaudio/s2-pro",
    "models_loaded": false,
    "engine_type": "none",
    "gpu_mem_free_gb": 15.5,
    "gpu_mem_total_gb": 15.6,
    "gpu_util_pct": 0.7,
    "gpu_ok": true,
    "session_id": "release-colab-external-images_20260319-060050_RC00",
    "gpu_vram_gb": 15.6
}


In [14]:
%%bash
set -e

# Install cloudflared if not present
which cloudflared >/dev/null 2>&1 || (
    python -m pip install cloudflared >/dev/null 2>&1 || true
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -O /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
)

# Kill any existing tunnel
pkill -f "cloudflared tunnel" 2>/dev/null || true
sleep 2

# Start tunnel
cloudflared tunnel --url http://127.0.0.1:8000 \
    --logfile /content/cloudflared.log \
    --metrics 0.0.0.0:9100 \
    > /content/cloudflared.out 2>&1 &

sleep 8
TUNNEL_URL=$(grep -o 'https://[-a-zA-Z0-9.]*trycloudflare.com' /content/cloudflared.out | head -n 1)
echo ""
echo "=========================================="
echo "TUNNEL URL: $TUNNEL_URL"
echo "=========================================="
echo ""
echo "Copy the URL above into Brainclip → Settings → Colab Tunnel URL"
echo ""
# Verify tunnel is live
curl -s "${TUNNEL_URL}/health" | python -m json.tool || echo "Health check failed — wait 10s and try again"



TUNNEL URL: https://documentary-reduced-saver-universe.trycloudflare.com

Copy the URL above into Brainclip → Settings → Colab Tunnel URL

{
    "status": "ok",
    "model": "fishaudio/s2-pro",
    "models_loaded": false,
    "engine_type": "none",
    "gpu_mem_free_gb": 15.5,
    "gpu_mem_total_gb": 15.6,
    "gpu_util_pct": 0.7,
    "gpu_ok": true,
    "session_id": "release-colab-external-images_20260319-060050_RC00",
    "gpu_vram_gb": 15.6
}


In [16]:
import os
from pathlib import Path

# Define the expected reference path
REF_DIR = Path('/content/cache/refs')
REF_DIR.mkdir(parents=True, exist_ok=True)

print(f'Current local reference files in {REF_DIR}:')
for f in REF_DIR.glob('*'):
    print(f'- {f.name}')

print('\nNote: If you are using a speaker name like "nova-spark", ensure a file named "nova-spark.wav" (or similar) exists in that folder, or provide a full URL.')

Current local reference files in /content/cache/refs:

Note: If you are using a speaker name like "nova-spark", ensure a file named "nova-spark.wav" (or similar) exists in that folder, or provide a full URL.


## Operational notes

- Paste the Cloudflare URL above into Brainclip **Settings → Colab Tunnel URL**.
- The runtime fetches the latest scripts directly from the Brainclip repository.
- Models stay loaded in memory across jobs. Speaker token cache lives in `/content/cache/refs/`.
- GPU memory is shown in the `/health` endpoint. If it drops below 2 GB free, restart the runtime.
- If you see `model_oom` errors, restart the Colab runtime (Runtime → Disconnect and delete runtime → Run again).